# Dataset Expansion Experiments, One Shard

Spec: `docs/specs/0007-dataset-expansion-experiments.md`

This notebook runs the three isolated expansion experiments over `brwac-clean-1.txt.gz`/the one-shard bronze dataset. Each experiment writes an independently reviewable dataset and the required CSV/notes artifacts under slugged directories.

```text
data/gold/experiments/<experiment_slug>/candidates
outputs/experiments/<experiment_slug>/candidate_counts.csv
outputs/experiments/<experiment_slug>/ground_vehicle_counts.csv
outputs/experiments/<experiment_slug>/vehicle_counts.csv
outputs/experiments/<experiment_slug>/review_sample.csv
outputs/experiments/<experiment_slug>/quality_notes.md
```


In [1]:
import os
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import tempfile
import zipfile

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")
SPARK_DRIVER_MEMORY = os.environ.get("TAL_QUAL_SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-dataset-expansion-experiments-one-shard")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(SRC_DIR.rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))
spark.sparkContext.addPyFile(str(package_zip))


In [3]:
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.como_article_ground_vehicle import (
    prefilter_como_article_ground_vehicle_bronze_dataframe,
    prepare_como_article_ground_vehicle_dataframe,
)
from tal_qual.dataset_expansion_experiments import (
    EXPERIMENT_SLUGS,
    accepted_candidates_dataframe,
    candidate_counts_dataframe,
    experiment_dataset_path,
    experiment_output_path,
    ground_vehicle_counts_dataframe,
    prefilter_dataset_expansion_bronze_dataframe,
    prepare_dataset_expansion_dataframe,
    review_sample_dataframe,
    vehicle_counts_dataframe,
    write_dataset_expansion_artifacts,
)


## Load One-Shard Bronze

The default paths match the spec. Override `TAL_QUAL_RAW_CORPUS_INPUT` or `TAL_QUAL_BRONZE_PATH` only when intentionally rerunning against a different shard/materialization.


In [4]:
# Use a dedicated spec-0007 bronze path by default so this exploratory notebook
# does not accidentally reuse a larger/full-corpus bronze materialization.
default_experiment_bronze_path = Path("data/bronze/brwac_segments_spec_0007_one_shard")
bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", str(default_experiment_bronze_path))
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", str(RAW_CORPUS_INPUT))

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path).persist()
bronze_rows = bronze_df.count()
print(f"Bronze path: {bronze_path}")
print(f"Raw path: {raw_path}")
print(f"Bronze rows: {bronze_rows:,}")
bronze_df.printSchema()


Bronze path: /home/jovyan/work/data/bronze/brwac_segments_spec_0007_one_shard
Raw path: /home/jovyan/work/data/raw/brwac-clean-1.txt.gz
Bronze rows: 4,673,057
root
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- text_original: string (nullable = true)
 |-- text_normalized: string (nullable = true)
 |-- match_text: string (nullable = true)



## Run Experiments

Each slug is independent: native Spark prefilter, Python extraction on plausible rows, Parquet candidate dataset, then the required review artifacts.


In [5]:
experiment_metrics = []
experiment_dataset_paths = {}
accepted_dataset_paths = {}

for slug in EXPERIMENT_SLUGS:
    print(f"\n=== {slug} ===")
    prefiltered_df = prefilter_dataset_expansion_bronze_dataframe(bronze_df, slug).persist()
    prefiltered_rows = prefiltered_df.count()
    print(f"Prefiltered bronze rows: {prefiltered_rows:,}")

    candidates_df = prepare_dataset_expansion_dataframe(prefiltered_df, slug).persist()
    candidate_rows = candidates_df.count()
    print(f"Candidate rows: {candidate_rows:,}")

    write_dataset_expansion_artifacts(candidates_df, slug)
    dataset_path = experiment_dataset_path(slug)
    accepted_path = dataset_path.parent / "accepted_candidates"
    accepted_candidates_dataframe(candidates_df).write.mode("overwrite").parquet(str(accepted_path))
    output_path = experiment_output_path(slug)
    experiment_dataset_paths[slug] = dataset_path
    accepted_dataset_paths[slug] = accepted_path
    print(f"Dataset: {dataset_path}")
    print(f"Accepted dataset: {accepted_path}")
    print(f"Review artifacts: {output_path}")

    accepted_df = accepted_candidates_dataframe(candidates_df).persist()
    accepted_rows = accepted_df.count()
    label_counts = {
        row["quality_label"]: row["count"]
        for row in candidate_counts_dataframe(candidates_df).collect()
    }
    false_positive_reason_rows = (
        candidates_df
        .where(F.col("quality_label").isin("review", "reject"))
        .groupBy("quality_reason")
        .count()
        .orderBy(F.col("count").desc(), "quality_reason")
        .limit(1)
        .collect()
    )
    top_false_positive_reason = false_positive_reason_rows[0]["quality_reason"] if false_positive_reason_rows else "none"
    accepted_rate = accepted_rows / candidate_rows if candidate_rows else 0.0

    if slug == "broader_ground_window_como_article" and accepted_rate >= 0.90:
        promotion_recommendation = "promote after baseline overlap check"
    elif slug == "bare_como_curated_ground" and accepted_rate >= 0.55:
        promotion_recommendation = "iterate cleanup, then reconsider"
    else:
        promotion_recommendation = "report-only for visualization build"

    experiment_metrics.append(
        {
            "experiment_slug": slug,
            "prefiltered_rows": prefiltered_rows,
            "candidate_rows": candidate_rows,
            "accepted_rows": accepted_rows,
            "distinct_grounds": accepted_df.select("ground_lemma").distinct().count(),
            "distinct_vehicle_heads": accepted_df.select("vehicle_head_clean").distinct().count(),
            "distinct_pairs": accepted_df.select("ground_lemma", "vehicle_head_clean").distinct().count(),
            "accepted_rate": accepted_rate,
            "top_false_positive_reason": top_false_positive_reason,
            "promotion_recommendation": promotion_recommendation,
        }
    )

    accepted_df.unpersist()
    candidates_df.unpersist()
    prefiltered_df.unpersist()

comparison_df = spark.createDataFrame(experiment_metrics)
comparison_output = REPO_ROOT / "outputs/experiments/comparison_table.csv"
(
    comparison_df
    .select(
        "experiment_slug",
        "prefiltered_rows",
        "candidate_rows",
        "accepted_rows",
        "distinct_grounds",
        "distinct_vehicle_heads",
        "distinct_pairs",
        "accepted_rate",
        "top_false_positive_reason",
        "promotion_recommendation",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(comparison_output))
)
comparison_df.orderBy(F.col("accepted_rows").desc()).show(truncate=False)



=== bare_como_curated_ground ===
Prefiltered bronze rows: 1,170


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Candidate rows: 1,184
Dataset: data/gold/experiments/bare_como_curated_ground/candidates
Accepted dataset: data/gold/experiments/bare_como_curated_ground/accepted_candidates
Review artifacts: outputs/experiments/bare_como_curated_ground

=== colloquial_que_nem_feito ===
Prefiltered bronze rows: 361
Candidate rows: 307
Dataset: data/gold/experiments/colloquial_que_nem_feito/candidates
Accepted dataset: data/gold/experiments/colloquial_que_nem_feito/accepted_candidates
Review artifacts: outputs/experiments/colloquial_que_nem_feito

=== broader_ground_window_como_article ===
Prefiltered bronze rows: 333
Candidate rows: 334
Dataset: data/gold/experiments/broader_ground_window_como_article/candidates
Accepted dataset: data/gold/experiments/broader_ground_window_como_article/accepted_candidates
Review artifacts: outputs/experiments/broader_ground_window_como_article
+------------------+-------------+--------------+----------------+--------------+----------------------+-----------------------

## Inspect Top Pairs

Use these compact tables for a first pass on whether the outputs are visualizable before opening the review samples.


In [6]:
for slug, dataset_path in accepted_dataset_paths.items():
    accepted_df = spark.read.parquet(str(dataset_path))
    print(f"\n=== {slug}: top accepted ground/vehicle pairs ===")
    ground_vehicle_counts_dataframe(accepted_df).show(30, truncate=False)



=== bare_como_curated_ground: top accepted ground/vehicle pairs ===
+------------------------+------------+------------------+-----+
|pattern_type            |ground_lemma|vehicle_head_clean|count|
+------------------------+------------+------------------+-----+
|bare_como_curated_ground|claro       |água              |8    |
|bare_como_curated_ground|escuro      |breu              |6    |
|bare_como_curated_ground|grande      |é                 |5    |
|bare_como_curated_ground|claro       |é                 |4    |
|bare_como_curated_ground|forte       |sempre            |4    |
|bare_como_curated_ground|preso       |suspeito          |4    |
|bare_como_curated_ground|triste      |eu                |4    |
|bare_como_curated_ground|baixo       |se                |3    |
|bare_como_curated_ground|claro       |essas             |3    |
|bare_como_curated_ground|claro       |funciona          |3    |
|bare_como_curated_ground|claro       |será              |3    |
|bare_como_curated_gr

## Inspect Review Samples

Each `review_sample.csv` is already written under `outputs/experiments/<slug>/`. The preview below keeps the notebook self-contained for quick judgment.


In [7]:
for slug, dataset_path in experiment_dataset_paths.items():
    candidates_df = spark.read.parquet(str(dataset_path))
    print(f"\n=== {slug}: review sample preview ===")
    review_sample_dataframe(candidates_df, limit=20).show(20, truncate=80)



=== bare_como_curated_ground: review sample preview ===
+----------------------------------------+------------------------+-----------+------------+--------------+--------------------------------------------------------------------------------+------------------+------------------+-------------+-----------------+------------+--------------------------------------------------------------------------------+------------------------------------------------------+----------------+----------+
|                            candidate_id|            pattern_type|ground_text|ground_lemma|connector_text|                                                                vehicle_text_raw|vehicle_text_clean|vehicle_head_clean|quality_label|   quality_reason|needs_review|                                                             candidate_full_text|                                           source_file|original_line_id|segment_id|
+----------------------------------------+------------------------+----

## Baseline Overlap And Dataset Strategy

The visualization build should start from a stable dataset, not from every exploratory branch. This section measures overlap against spec 0006 and writes a compact strategy table for the build phase.


In [8]:
print("Building spec-0006 one-shard baseline for overlap measurement...")
baseline_prefiltered_df = prefilter_como_article_ground_vehicle_bronze_dataframe(bronze_df).persist()
baseline_prefiltered_rows = baseline_prefiltered_df.count()
baseline_df = prepare_como_article_ground_vehicle_dataframe(baseline_prefiltered_df).persist()
baseline_rows = baseline_df.count()
baseline_pairs_df = baseline_df.select(
    "ground_lemma",
    F.col("vehicle_head_lemma").alias("vehicle_head_clean"),
).distinct().persist()
baseline_pair_count = baseline_pairs_df.count()
print(f"Spec-0006 prefiltered rows: {baseline_prefiltered_rows:,}")
print(f"Spec-0006 candidates: {baseline_rows:,}")
print(f"Spec-0006 distinct pairs: {baseline_pair_count:,}")

overlap_metrics = []
for row in comparison_df.collect():
    slug = row["experiment_slug"]
    accepted_df = spark.read.parquet(str(accepted_dataset_paths[slug])).persist()
    accepted_pairs_df = accepted_df.select("ground_lemma", "vehicle_head_clean").distinct().persist()
    accepted_pair_count = accepted_pairs_df.count()
    new_pairs = accepted_pairs_df.join(baseline_pairs_df, ["ground_lemma", "vehicle_head_clean"], "left_anti").count()
    overlapping_pairs = accepted_pair_count - new_pairs

    if slug == "broader_ground_window_como_article":
        strategy = "use now: high precision, small enough to review, article-como compatible"
    elif slug == "bare_como_curated_ground":
        strategy = "do not use for first viz: keep as next expansion after article-head cleanup"
    else:
        strategy = "do not use for first viz: split/redesign connector experiment"

    overlap_metrics.append(
        {
            "experiment_slug": slug,
            "accepted_rows": row["accepted_rows"],
            "accepted_rate": row["accepted_rate"],
            "accepted_distinct_pairs": accepted_pair_count,
            "baseline_overlapping_pairs": overlapping_pairs,
            "new_pairs_vs_baseline": new_pairs,
            "dataset_strategy": strategy,
        }
    )
    accepted_pairs_df.unpersist()
    accepted_df.unpersist()

strategy_df = spark.createDataFrame(overlap_metrics)
strategy_output = REPO_ROOT / "outputs/experiments/dataset_strategy.csv"
(
    strategy_df
    .orderBy(F.col("accepted_rows").desc())
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(strategy_output))
)
strategy_df.orderBy(F.col("accepted_rows").desc()).show(truncate=False)

baseline_pairs_df.unpersist()
baseline_df.unpersist()
baseline_prefiltered_df.unpersist()


Building spec-0006 one-shard baseline for overlap measurement...
Spec-0006 prefiltered rows: 333
Spec-0006 candidates: 204
Spec-0006 distinct pairs: 113
+-----------------------+------------------+-------------+--------------------------+---------------------------------------------------------------------------+----------------------------------+---------------------+
|accepted_distinct_pairs|accepted_rate     |accepted_rows|baseline_overlapping_pairs|dataset_strategy                                                           |experiment_slug                   |new_pairs_vs_baseline|
+-----------------------+------------------+-------------+--------------------------+---------------------------------------------------------------------------+----------------------------------+---------------------+
|608                    |0.5920608108108109|701          |3                         |do not use for first viz: keep as next expansion after article-head cleanup|bare_como_curated_ground     

DataFrame[source_file: string, original_line_id: bigint, segment_id: int, text_original: string, text_normalized: string, match_text: string]

## Build-Phase Dataset Choice

For the first visualization build, use the existing spec-0006 dataset plus `broader_ground_window_como_article/accepted_candidates` only if the overlap table shows meaningful incremental pairs. Keep `bare_como_curated_ground` and `colloquial_que_nem_feito` as report-only experiment outputs until their cleanup rules improve.


## Output Index


In [9]:
for slug in EXPERIMENT_SLUGS:
    print(slug)
    print(f"  dataset: {experiment_dataset_path(slug)}")
    print(f"  artifacts: {experiment_output_path(slug)}")
print("comparison: outputs/experiments/comparison_table.csv")
print("strategy: outputs/experiments/dataset_strategy.csv")


bare_como_curated_ground
  dataset: data/gold/experiments/bare_como_curated_ground/candidates
  artifacts: outputs/experiments/bare_como_curated_ground
colloquial_que_nem_feito
  dataset: data/gold/experiments/colloquial_que_nem_feito/candidates
  artifacts: outputs/experiments/colloquial_que_nem_feito
broader_ground_window_como_article
  dataset: data/gold/experiments/broader_ground_window_como_article/candidates
  artifacts: outputs/experiments/broader_ground_window_como_article
comparison: outputs/experiments/comparison_table.csv
strategy: outputs/experiments/dataset_strategy.csv
